# Databricks Medallion Architecture - Step 03: Gold & RPT Transformation Orchestration
This notebook populates the Star-Schema Data Model in `gold_db` (Fact & Dimension Delta Tables) and builds the business aggregation tables/views in `rpt_db` from `silver_db`.



In [ ]:
# Step 1: Create Gold Layer Dimensions & Fact Tables
# 1. dim_customers
spark.sql("""
CREATE OR REPLACE TABLE gold_db.dim_customers AS
SELECT 
    c.customer_id,
    c.customer_name,
    MIN(o.ordered_at) AS first_order_at,
    MAX(o.ordered_at) AS last_order_at,
    COUNT(o.order_id) AS total_orders_count,
    COALESCE(SUM(o.order_total_cents), 0) AS total_spend_cents
FROM silver_db.customers c
LEFT JOIN silver_db.orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
""")
print("Gold Table Created: gold_db.dim_customers")

# 2. dim_products
spark.sql("""
CREATE OR REPLACE TABLE gold_db.dim_products AS
SELECT 
    sku AS product_sku,
    product_name,
    product_type,
    price_cents,
    CAST(price_cents / 100.0 AS DECIMAL(10,2)) AS price_usd,
    description
FROM silver_db.products
""")
print("Gold Table Created: gold_db.dim_products")

# 3. dim_stores
spark.sql("""
CREATE OR REPLACE TABLE gold_db.dim_stores AS
SELECT 
    store_id,
    store_name,
    opened_at,
    tax_rate
FROM silver_db.stores
""")
print("Gold Table Created: gold_db.dim_stores")

# 4. dim_supplies
spark.sql("""
CREATE OR REPLACE TABLE gold_db.dim_supplies AS
SELECT 
    supply_id,
    supply_name,
    cost_cents,
    CAST(cost_cents / 100.0 AS DECIMAL(10,2)) AS cost_usd,
    is_perishable,
    product_sku
FROM silver_db.supplies
""")
print("Gold Table Created: gold_db.dim_supplies")

# 5. fact_orders
spark.sql("""
CREATE OR REPLACE TABLE gold_db.fact_orders AS
SELECT 
    order_id,
    customer_id,
    store_id,
    ordered_at,
    CAST(ordered_at AS DATE) AS order_date,
    subtotal_cents,
    tax_paid_cents,
    order_total_cents,
    CAST(subtotal_cents / 100.0 AS DECIMAL(10,2)) AS subtotal_usd,
    CAST(tax_paid_cents / 100.0 AS DECIMAL(10,2)) AS tax_paid_usd,
    CAST(order_total_cents / 100.0 AS DECIMAL(10,2)) AS order_total_usd
FROM silver_db.orders
""")
print("Gold Table Created: gold_db.fact_orders")

# 6. fact_order_items
spark.sql("""
CREATE OR REPLACE TABLE gold_db.fact_order_items AS
SELECT 
    item_id,
    order_id,
    sku AS product_sku
FROM silver_db.order_items
""")
print("Gold Table Created: gold_db.fact_order_items")



In [ ]:
# Step 2: Create RPT Layer Reporting Tables

# 1. rpt_monthly_store_performance
spark.sql("""
CREATE OR REPLACE TABLE rpt_db.rpt_monthly_store_performance AS
SELECT 
    DATE_FORMAT(fo.ordered_at, 'yyyy-MM') AS year_month,
    ds.store_id,
    ds.store_name,
    COUNT(fo.order_id) AS total_orders,
    SUM(fo.order_total_usd) AS total_revenue_usd,
    CAST(AVG(fo.order_total_usd) AS DECIMAL(10,2)) AS average_order_value_usd,
    SUM(fo.tax_paid_usd) AS total_tax_collected_usd
FROM gold_db.fact_orders fo
JOIN gold_db.dim_stores ds ON fo.store_id = ds.store_id
GROUP BY DATE_FORMAT(fo.ordered_at, 'yyyy-MM'), ds.store_id, ds.store_name
ORDER BY year_month DESC, total_revenue_usd DESC
""")
print("RPT Table Created: rpt_db.rpt_monthly_store_performance")

# 2. rpt_customer_lifetime_value
spark.sql("""
CREATE OR REPLACE TABLE rpt_db.rpt_customer_lifetime_value AS
SELECT 
    dc.customer_id,
    dc.customer_name,
    CAST(dc.first_order_at AS DATE) AS first_order_date,
    CAST(dc.last_order_at AS DATE) AS last_order_date,
    dc.total_orders_count AS total_orders,
    CAST(dc.total_spend_cents / 100.0 AS DECIMAL(12,2)) AS total_lifetime_spend_usd,
    CASE 
        WHEN dc.total_orders_count > 0 THEN CAST((dc.total_spend_cents / 100.0) / dc.total_orders_count AS DECIMAL(10,2))
        ELSE 0.00
    END AS avg_order_value_usd,
    CASE 
        WHEN (dc.total_spend_cents / 100.0) >= 100 THEN 'VIP Customer'
        WHEN (dc.total_spend_cents / 100.0) >= 30 THEN 'Regular Customer'
        WHEN (dc.total_spend_cents / 100.0) > 0 THEN 'Occasional Customer'
        ELSE 'No Purchases'
    END AS customer_segment
FROM gold_db.dim_customers dc
ORDER BY total_lifetime_spend_usd DESC
""")
print("RPT Table Created: rpt_db.rpt_customer_lifetime_value")

# 3. rpt_product_sales_summary
spark.sql("""
CREATE OR REPLACE TABLE rpt_db.rpt_product_sales_summary AS
SELECT 
    dp.product_sku,
    dp.product_name,
    dp.product_type,
    COUNT(foi.item_id) AS total_units_sold,
    CAST(COUNT(foi.item_id) * dp.price_usd AS DECIMAL(12,2)) AS gross_revenue_usd,
    DENSE_RANK() OVER (ORDER BY COUNT(foi.item_id) * dp.price_usd DESC) AS product_rank_by_revenue
FROM gold_db.dim_products dp
LEFT JOIN gold_db.fact_order_items foi ON dp.product_sku = foi.product_sku
GROUP BY dp.product_sku, dp.product_name, dp.product_type, dp.price_usd
ORDER BY product_rank_by_revenue ASC
""")
print("RPT Table Created: rpt_db.rpt_product_sales_summary")



In [ ]:
# Step 3: Verify Gold & RPT Layer Tables
print("=== RPT: Monthly Store Performance ===")
display(spark.sql("SELECT * FROM rpt_db.rpt_monthly_store_performance LIMIT 10"))

print("=== RPT: Customer Lifetime Value (Top 10) ===")
display(spark.sql("SELECT * FROM rpt_db.rpt_customer_lifetime_value LIMIT 10"))

print("=== RPT: Product Sales Summary ===")
display(spark.sql("SELECT * FROM rpt_db.rpt_product_sales_summary LIMIT 10"))

